# 01 Data audit

Load the raw file, print its real schema, check units, price formats, dates and missing values, then run the
documented cleaning rules and look at the funnel. Nothing here assumes column names: the mapping in
`config.COLUMN_ALIASES` is printed so it can be checked against the file.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from house_prices import clean, config, load

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", context="notebook")
config.PATHS.figures_dir.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name):
    fig.savefig(config.PATHS.figures_dir / f"{name}.png", dpi=140, bbox_inches="tight")

## Load and inspect the raw file

In [ ]:
raw = load.load_raw()
print(load.schema_report(raw))
print()
print(load.mapping_report(raw))
synthetic = load.is_synthetic(load.apply_schema(raw))
print()
print("Synthetic sample file:", synthetic)

## Canonical view of the columns the project uses

In [ ]:
canonical = load.apply_schema(raw)
canonical.head()

## Purpose, property types and cities

Only `For Sale` listings are modelled; rentals are a different problem with different price scales.

In [ ]:
print(canonical["purpose"].value_counts(dropna=False).to_string())
print()
print(canonical["property_type"].value_counts(dropna=False).to_string())
print()
print(canonical["city"].value_counts(dropna=False).to_string())

## Area units and price formats

In [ ]:
print(canonical["area_unit"].value_counts(dropna=False).to_string())
print()
text_prices = clean.units.count_text_prices(canonical["price"])
print(f"Price column dtype: {canonical['price'].dtype}; rows stored as text: {text_prices:,}")
if text_prices:
    display(canonical.loc[pd.to_numeric(canonical["price"], errors="coerce").isna(), "price"].head(10))

## Dates

`date_added` is parsed month-first. If any value has a first component above 12 the assumption is wrong and
`clean.parse_dates` must be changed.

In [ ]:
parsed_dates = clean.parse_dates(canonical["date_added"])
first_part = canonical["date_added"].astype(str).str.extract(r"^(\d+)[/-]", expand=False).astype(float)
print("Unparseable dates:", int(parsed_dates.isna().sum()))
print("Earliest:", parsed_dates.min(), " Latest:", parsed_dates.max())
print("Rows whose first component exceeds 12 (would contradict month-first):", int((first_part > 12).sum()))

## Missing values in the modelling columns

In [ ]:
cols = ["property_type", "price", "location", "city", "latitude", "longitude", "baths", "bedrooms", "date_added"]
missing = canonical[cols].isna().mean().sort_values(ascending=False) * 100
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(x=missing.values, y=missing.index, ax=ax, color="#4c78a8")
ax.set_xlabel("Missing (%)")
ax.set_title("Missing values by column (raw file)")
save_fig(fig, "audit_missing")
plt.show()

## Run the cleaning rules

In [ ]:
settings = config.CleaningSettings()
cleaned, funnel, info, parsed = clean.clean(canonical, settings)
funnel_df = pd.DataFrame(funnel)[["key", "removed", "modified", "remaining", "description"]]
display(funnel_df)
print()
print(f"Sq ft per Marla: {info['marla_sqft']:.2f} ({info['marla_source']})")
print(f"Raw rows {len(raw):,} -> cleaned rows {len(cleaned):,} ({len(cleaned) / len(raw):.1%})")

In [ ]:
steps = funnel_df[funnel_df["removed"] > 0].sort_values("removed")
fig, ax = plt.subplots(figsize=(8, 0.45 * len(steps) + 1.5))
sns.barplot(x="removed", y="key", data=steps, ax=ax, color="#e45756")
ax.set_xlabel("Rows removed")
ax.set_ylabel("")
ax.set_title("Rows removed by each cleaning rule")
save_fig(fig, "audit_funnel")
plt.show()

## Price per sq ft before and after the outlier rule

The per-city IQR fence on log price per sq ft is applied last. The plot shows what it removes.

In [ ]:
before = parsed[(parsed["purpose"] == config.PURPOSE_SALE) & (parsed["price"] > 0) & (parsed["area_sqft"] > 0)].copy()
before["ppsf"] = before["price"] / before["area_sqft"]
after = cleaned.assign(ppsf=cleaned["price"] / cleaned["area_sqft"])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, frame, title in zip(axes, (before, after), ("Before outlier rule", "After cleaning")):
    sns.boxplot(data=frame, x="city", y="ppsf", ax=ax, color="#9ecae9", fliersize=1)
    ax.set_yscale("log")
    ax.set_title(title)
    ax.set_xlabel("")
axes[0].set_ylabel("PKR per sq ft (log scale)")
save_fig(fig, "audit_ppsf_before_after")
plt.show()

## Saved report

In [ ]:
report = config.PATHS.data_quality_md
print(report.read_text(encoding="utf-8")[:1500] if report.exists() else "Run `python -m house_prices.clean` first.")